3D Shape Classifier
(Heyrana, Quimno - 2024)

Imports and Setup

In [ ]:
# imports
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import logging
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import json
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

Data Augmentation

In [ ]:
class PointCloudAugmentor:
    def __init__(
        self,
        jitter_sigma: float = 0.02,
        jitter_clip: float = 0.05,
        scale_low: float = 0.8,
        scale_high: float = 1.25,
        translate_range: float = 0.2
    ):
        self.jitter_sigma = jitter_sigma
        self.jitter_clip = jitter_clip
        self.scale_low = scale_low
        self.scale_high = scale_high
        self.translate_range = translate_range

    def random_jitter(self, points: np.ndarray) -> np.ndarray:
        noise = np.clip(
            np.random.normal(0, self.jitter_sigma, points.shape),
            -self.jitter_clip,
            self.jitter_clip
        )
        return points + noise

    def random_scale(self, points: np.ndarray) -> np.ndarray:
        scale = np.random.uniform(self.scale_low, self.scale_high)
        return points * scale

    def random_translate(self, points: np.ndarray) -> np.ndarray:
        translation = np.random.uniform(-self.translate_range, self.translate_range, 3)
        return points + translation

    def __call__(self, points: np.ndarray) -> np.ndarray:
        points = self.random_jitter(points.copy())
        points = self.random_scale(points)
        points = self.random_translate(points)
        return points

Dataset Loader

In [ ]:
class ShapeNetDataset(Dataset):
    def __init__(
        self,
        root_dir: str,
        split: str = 'train',
        num_points: int = 1024,
        augment: bool = True,
        num_augmentations: int = 9  # Augmentations per scan
    ):
        self.root_dir = root_dir
        self.split = split
        self.num_points = num_points
        self.augmentor = PointCloudAugmentor() if augment else None
        self.num_augmentations = num_augmentations if augment else 0
        self.class_folders = ['sphere', 'cone', 'cube', 'cylinder', 'cuboid']
        
        logger.debug(f"Initializing dataset with root_dir: {root_dir}, split: {split}")
        self.setup_data()

    def setup_data(self):
        """Load and prepare the dataset, with detailed logging."""
        data, labels = [], []
        
        # Load data for the current split
        split_dir = os.path.join(self.root_dir, self.split)
        logger.debug(f"Looking for data in: {split_dir}")

        for label, class_name in enumerate(self.class_folders):
            class_path = os.path.join(split_dir, class_name)
            if not os.path.exists(class_path):
                continue

            files = [f for f in os.listdir(class_path) if f.endswith('.TXT')]
            logger.debug(f"Found {len(files)} files in {class_path}")

            for file in files:
                file_path = os.path.join(class_path, file)
                try:
                    points = np.loadtxt(file_path, delimiter=',', skiprows=3)
                    
                    if len(points.shape) != 2 or points.shape[1] != 3:
                        continue
                        
                    # Normalize points
                    centroid = np.mean(points, axis=0)
                    points -= centroid
                    dist = np.max(np.sqrt(np.sum(points ** 2, axis=1)))
                    if dist > 0:
                        points /= dist

                    # Sample points if needed
                    if len(points) > self.num_points:
                        indices = np.random.choice(len(points), self.num_points, replace=False)
                        points = points[indices]
                    elif len(points) < self.num_points:
                        indices = np.random.choice(len(points), self.num_points, replace=True)
                        points = points[indices]
                    
                    # Add original points
                    data.append(points)
                    labels.append(label)
                    
                    # Add augmented versions if in training split
                    if self.split == 'train' and self.augmentor:
                        for _ in range(self.num_augmentations):
                            augmented_points = self.augmentor(points.copy())
                            data.append(augmented_points)
                            labels.append(label)
                            
                        logger.debug(f"Created {self.num_augmentations} augmentations for {file}")
                    
                except Exception as e:
                    logger.error(f"Error processing {file_path}: {str(e)}")
                    continue

        if not data:
            raise ValueError(f"No valid data files found in {split_dir}")

        self.data = np.array(data)
        self.labels = np.array(labels)
        
        # Log dataset statistics
        total_original = len(files) if files else 0
        total_augmented = len(self.data) - total_original
        logger.info(f"\nDataset statistics for {self.split} split:")
        logger.info(f"Original samples: {total_original}")
        logger.info(f"Augmented samples: {total_augmented}")
        logger.info(f"Total samples: {len(self.data)}")
        
        # Log per-class statistics
        unique_labels, counts = np.unique(self.labels, return_counts=True)
        for label, count in zip(unique_labels, counts):
            logger.info(f"Class {self.class_folders[label]}: {count} samples "
                      f"({count/(1 + self.num_augmentations):.0f} original + "
                      f"{count - count/(1 + self.num_augmentations):.0f} augmented)")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        points = self.data[idx]
        label = self.labels[idx]
        return torch.from_numpy(points).float(), label